### Tools
#### Models can request to call tools that performs such as fetching data from the database , searching the web or running the code , tools are pairing of: 1. A schema , including the name of the tool, a discription and/or argument definition(often a JSON schema), 2. A function or coroutine to excecute.

In [5]:
from langchain_groq import ChatGroq

model = ChatGroq(model='llama-3.1-8b-instant')

In [6]:
from langchain.tools import tool
@tool
def get_weather(city:str)->str:
    """Get the weather at the location"""
    return f"It's sunny in {city}"
model_with_tool = model.bind_tools([get_weather])

In [8]:
response = model_with_tool.invoke("what is the weather in bangalore")
for tool_call in response.tool_calls:
    # View tool call made by the model
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

Tool:get_weather
Args:{'city': 'bangalore'}


In [ ]:
# Onother way of creating tools
# create agent
from langchain.agents import create_agent

def get_weather(city:str)->str:
    '''Get the weather for a city.'''
    return f"The weather in a {city} is sunny."

agent = create_agent(
    model = "gpt-5",
    tools = [get_weather],
    system_prompt = "You are a helpful assistant."
)
agent

### tool Execution loop

In [11]:
# step 1 : model generates tool call 
messages =[{"role":"user", "content":"what is the weather in bangalore"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)
print(messages)
# step 2 : excecutes tools and collect result
for tool_call in ai_msg.tool_calls:
    # execute the tool with the generated argument
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
print(messages)
# step 3: pass result back to the model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.text)


[{'role': 'user', 'content': 'what is the weather in bangalore'}, AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'bqh79e2c4', 'function': {'arguments': '{"city":"bangalore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 219, 'total_tokens': 234, 'completion_time': 0.025202846, 'completion_tokens_details': None, 'prompt_time': 0.018365703, 'prompt_tokens_details': None, 'queue_time': 0.053509406, 'total_time': 0.043568549}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eacc1-4858-73d0-b016-9ed4a3a64afd-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'bangalore'}, 'id': 'bqh79e2c4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'output_tokens': 15, 'total_tokens': 234})]
[{'role': 'user', 'content': 